In [1]:
!git clone --branch reim https://github.com/baohuyvanba/hcmus_cvi_cp-vton.git

Cloning into 'hcmus_cvi_cp-vton'...
remote: Enumerating objects: 297, done.
remote: Counting objects: 100% (171/171), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 297 (delta 94), reused 151 (delta 78), pack-reused 126
Receiving objects: 100% (297/297), 278.79 MiB | 35.59 MiB/s, done.
Resolving deltas: 100% (161/161), done.


In [2]:
%cd /kaggle/working/hcmus_cvi_cp-vton

/kaggle/working/hcmus_cvi_cp-vton


In [3]:
%%writefile cp_dataset.py
# coding=utf-8
import torch
import torch.utils.data as data
import torchvision.transforms as transforms
from torch.utils.data import DistributedSampler

from PIL import Image
from PIL import ImageDraw

import os
import numpy as np
import json


class CPDataset(data.Dataset):
    """
    Dataset class for input (CP-VTON dataset)
    """
    def __init__(self, opt):
        super(CPDataset, self).__init__()
        self.opt = opt
        self.root = opt.dataroot
        self.datamode = opt.datamode
        self.stage = opt.stage
        self.data_list = f"{opt.datamode}_pairs.txt"
        self.fine_height = opt.fine_height
        self.fine_width  = opt.fine_width
        self.radius = opt.radius
        self.data_path = os.path.join(opt.dataroot, opt.datamode)
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize( #Normalize: (Each value of tensor - mean)/std
                mean=(0.5,),
                std=(0.5,)
            )
        ])

        #read data list
        img_names = []
        cth_names = []

        with open(os.path.join(opt.dataroot, self.data_list), 'r') as f:
            for line in f.readlines():
                #Form: image_1_name.txt cloth_1_name.txt ...
                img_name, cth_name = line.strip().split()
                img_names.append(img_name)
                cth_names.append(cth_name)

        self.img_names = img_names
        self.cth_names = cth_names

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, index):
        cth_name = self.cth_names[index] #Cloth image's name
        img_name = self.img_names[index] #Person image's name

        #Note - Pytorch image: (channel x height x width)

        #CLOTH and CLOTH Mask in GMM and TOM
        if self.stage == 'GMM':
            cloth = Image.open(os.path.join(self.data_path, 'cloth', cth_name))
            clothmask = Image.open(os.path.join(self.data_path, 'cloth-mask', cth_name))
        else: #self.stage == 'TOM'
            cloth = Image.open(os.path.join(self.data_path, 'warped-cloth', cth_name))
            clothmask = Image.open(os.path.join(self.data_path, 'warped-mask', cth_name))

        #CLOTH: read
        #cloth = Image.open(os.path.join(self.data_path, 'cloth', cth_name))
        cloth = self.transform(cloth)
        if (cloth.shape[1], cloth.shape[2]) != (256, 192):
            cloth = transforms.Resize((256, 192))(cloth) #256x192x3 -> 3x256x192


        #CLOTH-MASK: read
        #clothmask = Image.open(os.path.join(self.data_path, 'cloth-mask', cth_name))
                    #This mask image only has 1 color channel
        cmask_array = np.array(clothmask)                       #1x256x192 -> 256x192
        cmask_array = (cmask_array >= 128).astype(np.float32)   #Binary mask: 0(background), 1(foreground/cloth region)
        clothmask = torch.from_numpy(cmask_array).unsqueeze(0)  #256x192 -> 1x256x192: add dim at 0 using unsqueeze because Pytorch image data is (channel x height x width)


        #IMAGE: read person image
        image = Image.open(os.path.join(self.data_path, 'image', img_name))
        image = self.transform(image) #256x192x3 -> 3x256x192 and normalize [0, 255] to [-1, 1]


        #IMAGE-PARSE: read parsing image
        parse_name = img_name.replace('.jpg', '.png')
        img_parse = Image.open(os.path.join(self.data_path, 'image-parse', parse_name))
                    #This is RGB image: 3x256x192
        parse_array = np.array(img_parse) #3x256x192
          #Binary shape parse: 1:all body, 0:background
        parse_shape = (parse_array > 0)
          #downsample shape parse
        parse_shape = Image.fromarray((parse_shape * 255).astype(np.uint8))
        parse_shape = parse_shape.resize((self.fine_width // 16, self.fine_height // 16))
        parse_shape = parse_shape.resize((self.fine_width, self.fine_height))
        person_shape = self.transform(parse_shape)     #to Tensor (1x256x192) then Normalize: [0,255] -> [-1,1]

          #Binary head mask: 1:head segment -> keep image identity
        parse_head = (parse_array == 1).astype(np.float32) + \
                     (parse_array == 2).astype(np.float32) + \
                     (parse_array == 4).astype(np.float32) + \
                     (parse_array == 13).astype(np.float32)
        person_head = torch.from_numpy(parse_head)      #to Tensor: {0,1}

          #Binary cloth mask: 1:upbody segment -> where to add cloth segment
        parse_cloth = (parse_array == 5).astype(np.float32) + \
                      (parse_array == 6).astype(np.float32) + \
                      (parse_array == 7).astype(np.float32)
        person_cthmask = torch.from_numpy(parse_cloth)  #to Tensor: {0,1}

          #DIFF: Binary rest of body mask (Neither cloth region nor background)
        restbody_mask = (parse_array == 1).astype(np.float32) + \
                        (parse_array == 2).astype(np.float32) + \
                        (parse_array == 3).astype(np.float32) + \
                        (parse_array == 4).astype(np.float32) + \
                        (parse_array > 7).astype(np.float32)
        restbody_mask = torch.from_numpy(restbody_mask).unsqueeze(0) #to Tensor: {0, 1}

        #Upper cloth: cloth region and  head region mask on image
        img_cthmask = image * person_cthmask + (1 - person_cthmask)  #Get pixels value of cloth region and fill 1 for other parts -> [-1, 1]*{0,1} + {1, 0} = [-1, 1]: Mask of cloth region on person image
        img_headmsk = image * person_head - (1 - person_head)        #Get pixels value of head region and fill 0 for other parts  -> [-1, 1]*{0,1} + {0, 1} = [-1, 1]: Mask of head region on person image

        #POSE: read
        pose_name = img_name.replace('.jpg', '_keypoints.json')
        with open(os.path.join(self.data_path, 'pose', pose_name), 'r') as f:
            pose_label = json.load(f)
            pose_data = pose_label['people'][0]['pose_keypoints']
            pose_data = np.array(pose_data)
            pose_data = pose_data.reshape((-1, 3))

        point_num = pose_data.shape[0]
        pose_map = torch.zeros(point_num, self.fine_height, self.fine_width)
        r = self.radius
        img_pose = Image.new('L', (self.fine_width, self.fine_height))
        pose_draw = ImageDraw.Draw(img_pose)
        for i in range(point_num):
            one_map = Image.new('L', (self.fine_width, self.fine_height))
            draw = ImageDraw.Draw(one_map)
            pointx = pose_data[i, 0]
            pointy = pose_data[i, 1]
            if pointx > 1 and pointy > 1:
                draw.rectangle((pointx - r, pointy - r, pointx + r, pointy + r), 'white', 'white')
                pose_draw.rectangle((pointx - r, pointy - r, pointx + r, pointy + r), 'white', 'white')
            one_map = self.transform(one_map)
            pose_map[i] = one_map[0]

        # just for visualization
        img_pose = self.transform(img_pose)

        #PERSON REPRESENTATION: Cloth-agnostic Representation: person shape, head region mask, pose map.
        agnostic = torch.cat([person_shape, img_headmsk, pose_map], 0)


        im_g = Image.open('grid.png')
        im_g = self.transform(im_g)

        result = {
            'c_name': cth_name,         # for visualization
            'im_name': img_name,        # for visualization or ground truth
            'cloth': cloth,             # for input
            'cloth_mask': clothmask,    # for input
            'image': image,             # for visualization
            'agnostic': agnostic,       # for input
            'parse_cloth': img_cthmask, # for ground truth
            'shape': person_shape,      # for visualization
            'head': img_headmsk,        # for visualization
            'restbody': restbody_mask,  #
            'pose_image': img_pose,     # for visualization
            'grid_image': im_g,         # for visualization
        }

        return result


class CPDataLoader(object):
    """
    This class creates a data loader for cp-vton dataset.

    Attributes:
        dataset (torch.utils.data.Dataset): The dataset to be loaded.
        data_loader (torch.utils.data.DataLoader): The data loader object.
        data_iter (iter): An iterator over the data loader.
    """
    def __init__(self, opt, dataset):
        super(CPDataLoader, self).__init__()
        self.dataset = dataset
        self.data_loader = torch.utils.data.DataLoader(
            dataset,
            batch_size=opt.batch_size,     #Define the batch size
            num_workers=opt.workers,       #Define the number of worker threads to use for data loading (main thread only)
            pin_memory=True,               #Improve data transfer's speed between CPUs and GPUs
            sampler=DistributedSampler(dataset)
        )
        self.data_iter = self.data_loader.__iter__()

    def next_batch(self):
        """
        Returns the next batch of data from the data loader.
        Returns:
            batch: A batch of data from the dataset.
        """
        try:
            batch = next(self.data_iter)
        except StopIteration:
            self.data_iter = self.data_loader.__iter__()
            batch = self.data_iter.__next__()
        return batch

Overwriting cp_dataset.py


In [4]:
%%writefile train.py
#coding=utf-8
import torch
import torch.nn as nn
import torch.nn.functional as F

import argparse
import os

from tqdm import tqdm
from cp_dataset import CPDataset, CPDataLoader                                      #file: cp_dataset.py
from networks import GMM, UnetGenerator, VGGLoss, load_checkpoint, save_checkpoint  #file: networks.py

from torch.utils.tensorboard import SummaryWriter
from visualization import board_add_image, board_add_images                         #file: visualization.py

import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

def get_opt():
    """
    This function defines the command-line arguments for the program.
    Returns: A namespace object (opt) containing all the parsed arguments and their values.
    """

    #Define arguments related to model and training: name, batch size, number of workers
    parser = argparse.ArgumentParser()
    parser.add_argument("--name", default = "GMM")
    parser.add_argument('-j', '--workers', type=int, default=2)
    parser.add_argument('-b', '--batch-size', type=int, default=4)
    parser.add_argument("--stage", default="GMM")
    #Define arguments related to data
    parser.add_argument("--dataroot", default = "data")
    parser.add_argument("--datamode", default = "train")
    parser.add_argument("--data_list", default = "train_pairs.txt")
    #Define arguments related to image processing
    parser.add_argument("--fine_width", type=int, default = 192)
    parser.add_argument("--fine_height", type=int, default = 256)
    parser.add_argument("--radius", type=int, default = 5)
    parser.add_argument("--grid_size", type=int, default = 5)
    # Define arguments related to training optimization
    parser.add_argument('--lr', type=float, default=0.0001, help='initial learning rate for adam')
    parser.add_argument('--tensorboard_dir', type=str, default='tensorboard', help='save tensorboard infos')
    parser.add_argument('--checkpoint_dir', type=str, default='checkpoints', help='save checkpoint infos')
    parser.add_argument('--checkpoint', type=str, default='', help='model checkpoint for initialization')
    parser.add_argument("--keep_step", type=int, default=100000)
    parser.add_argument("--decay_step", type=int, default=100000)
    parser.add_argument("--save_count", type=int, default=100)
    #Define arguments related to data training visualization and logging
    parser.add_argument("--display_count", type=int, default = 20)
    #Shuffling data
    parser.add_argument("--shuffle", action='store_true', help='shuffle input data')

    opt = parser.parse_args()
    return opt

def train_gmm(opt, train_loader, model, board, device_id):
    """
    This function run the train processs for the GMM model on the provided data loader get from CP-VTON dataset.

    Args:
        opt (argparse.Namespace): Namespace containing the parsed command-line arguments.
        train_loader (DataLoader): Data loader for training data.
        model (nn.Module): The GMM model to be trained.
        board (object): Object for logging information (e.g., TensorBoard).
    """
    gpus_id = device_id
    #Create model and move to GPU: Distributed Data Parallel (DDP) for multi-GPU training
    model = DDP(model.to(gpus_id), [gpus_id])
    model.train()

    #Define loss function: L1 Loss
    criterionL1 = nn.L1Loss()
    #Define optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=opt.lr, betas=(0.5, 0.999))

    #Training loop process
    for epoch in tqdm(range(opt.keep_step + opt.decay_step)):
        #Set epoch for the data sampler
        train_loader.data_loader.sampler.set_epoch(epoch)
        #Get a batch of data from dataset
        inputs = train_loader.next_batch()

        #Move data to GPUs for training
        image        = inputs['image'].to(gpus_id)      # [4, 3, 256, 192]
        img_pose     = inputs['pose_image'].to(gpus_id) # [4, 1, 256, 192]
        img_headmsk  = inputs['head'].to(gpus_id)       # [4, 3, 256, 192]
        person_shape = inputs['shape'].to(gpus_id)      # [4, 1, 256, 192]
        agnostic     = inputs['agnostic'].to(gpus_id)   # [4, 22, 256, 192]
        cloth        = inputs['cloth'].to(gpus_id)      # [4, 3, 256, 192]
        cthmask      = inputs['cloth_mask'].to(gpus_id) # [4, 1, 256, 192]
        img_cthmask  = inputs['parse_cloth'].to(gpus_id)# [4, 3, 256, 192] #ground_truth for warp cloth mask
        restbody_msk = inputs['restbody'].to(gpus_id)    # DIFF: add rest of the body mask
        img_grid     = inputs['grid_image'].to(gpus_id) # [4, 3, 256, 192]

        #Forward pass through the model
        grid, theta = model(agnostic, cloth)

        #Warp cloth and mask using the predicted grid, warped_grid use for visualization
        warped_cloth = F.grid_sample(cloth, grid, padding_mode='border', align_corners=True)
        warped_mask = F.grid_sample(cthmask, grid, padding_mode='zeros', align_corners=True)
        warped_grid = F.grid_sample(img_grid, grid, padding_mode='zeros', align_corners=True)
        #DIFF: person with warped cloth and person with ground truth cloth
        warped_person = restbody_msk*image  + (1 - restbody_msk)*warped_cloth
        ground_person = restbody_msk*image  + (1 - restbody_msk)*img_cthmask 

        #Visualizations for logging
        visuals = [[img_headmsk, person_shape, img_pose],          #head, shape, pose
                   [cloth, warped_cloth, img_cthmask],             #cloth, warped cloth, parsed cloth images
                   [warped_grid, (warped_cloth+image)*0.5, image]] #warped grid, combine image with warped_cloth, input image

        #Calculate loss with L1 Loss: 0.5(between warped cloth and ground truth) + (warped person and ground truth) DIFF
        loss = 0.5*criterionL1(warped_cloth, img_cthmask) + criterionL1(warped_person, ground_person)

        #Backpropagation and optimization
        optimizer.zero_grad()  #Clear gradients
        loss.backward()        #Backpropagate loss
        optimizer.step()       #Update model weights

        #Logging (every number of steps, which was defined with argument before)
        if (epoch+1) % opt.display_count == 0:
            board_add_images(board, 'combine', visuals, epoch+1)
            board.add_scalar('metric', loss.item(), epoch+1)
        #Save weight after number of steps
        if (epoch+1) % opt.save_count == 0:
            save_checkpoint(model, os.path.join(opt.checkpoint_dir, opt.name, 'step_%06d.pth' % (epoch+1)))

def train_tom(opt, train_loader, model, board, device_id):
    """
    This function run the train processs for the TOM model on the provided data loader.
    Args:
        opt (argparse.Namespace): Namespace containing the parsed command-line arguments.
        train_loader (DataLoader): Data loader for training data.
        model (nn.Module): The TOM model to be trained.
        board (object): Object for logging information (e.g., TensorBoard).
    """
    gpus_id = device_id
    # Create model and move to GPU: Distributed Data Parallel (DDP) for multi-GPU training
    model = DDP(model.to(gpus_id), [gpus_id])
    model.train()
    
    #Define loss function: 2 L1 Loss, 1 VGGLoss
    criterionL1  = nn.L1Loss()   #for measuring pixel-wise difference
    criterionVGG  = VGGLoss()    #VGG-based perceptual loss
    criterionMask = nn.L1Loss()  #for mask
    criterionAdv = torch.nn.BCELoss() #DIFF

    #Define optimizer: Adam optimizer with defined learning rate and betas
    optimizer = torch.optim.Adam(model.parameters(), lr=opt.lr, betas=(0.5, 0.999))
    #Learning rate scheduler: keep in keep_step, and decrease in decay_step
    # scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda = lambda step: 1.0 -
    #             max(0, step - opt.keep_step) / float(opt.decay_step + 1))

    #Training loop process
    for step in tqdm(range(opt.keep_step + opt.decay_step)):
        #Set epoch for the data sampler
        train_loader.data_loader.sampler.set_epoch(step)
        #Get a batch of data from dataset
        inputs = train_loader.next_batch()
            
        image        = inputs['image'].to(gpus_id)      # [4, 3, 256, 192]
        img_pose     = inputs['pose_image']             # [4, 1, 256, 192]
        img_headmsk  = inputs['head']                   # [4, 3, 256, 192]
        person_shape = inputs['shape']                  # [4, 1, 256, 192]
        agnostic     = inputs['agnostic'].to(gpus_id)   # [4, 22, 256, 192]
        cloth        = inputs['cloth'].to(gpus_id)      # [4, 3, 256, 192]
        cthmask      = inputs['cloth_mask'].to(gpus_id) # [4, 1, 256, 192]

        #Forward pass through the model
        outputs = model(torch.cat([agnostic, cloth],1))

        #Split the model output into predicted rendered image and mask composite
        p_rendered, m_composite = torch.split(outputs, 3,1)
        #Apply activation functions: tanh and sigmoid
        p_rendered  = torch.tanh(p_rendered)
        m_composite = torch.sigmoid(m_composite)

        #Combine predicted rendered image and mask for final try-on image
        p_tryon = cloth*m_composite + p_rendered*(1 - m_composite)

        #Visualizations for logging
        visuals = [[img_headmsk, person_shape, img_pose], # Head, shape, and pose images
                   [cloth, cthmask*2-1, m_composite*2-1],  # Cloth, scaled cloth mask, and scaled mask composite
                   [p_rendered, p_tryon, image]]           # Predicted rendered image, try-on image, and input image

        #Calculate loss using loss function
        loss_l1   = criterionL1(p_tryon, image).to(device_id)           #L1 loss between try-on and input image
        loss_vgg  = criterionVGG(p_tryon, image).to(device_id)          #VGGLoss between try-on image and image (ground truth) (perceptual similarity)
        loss_mask = criterionMask(m_composite, cthmask).to(device_id)   #L1 loss between mask composite (predicted) and cloth mask (ground truth)

        #DIFF
        # loss_adv = criterionAdv(dis_fake, real)

        #Combine all losses
        loss = (loss_l1 + loss_vgg + loss_mask).to(device_id)

        #Backpropagation and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        #Logging: same with train_gmm
        if (step+1) % opt.display_count == 0:
            board_add_images(board, 'combine', visuals, step+1)
            board.add_scalar('metric', loss.item(), step+1)
            board.add_scalar('L1', loss_l1.item(), step+1)
            board.add_scalar('VGG', loss_vgg.item(), step+1)
            board.add_scalar('MaskL1', loss_mask.item(), step+1)
        #Save weight: same with train_gmm
        if (step+1) % opt.save_count == 0:
            save_checkpoint(model, os.path.join(opt.checkpoint_dir, opt.name, 'step_%06d.pth' % (step+1)))

def main():
    dist.init_process_group(backend='nccl')
    torch.cuda.manual_seed_all(244)
    rank = dist.get_rank()
    device_id = rank % torch.cuda.device_count()
    torch.cuda.set_device(device_id)

    opt = get_opt()
    print("Start to train stage: %s, named: %s!" % (opt.stage, opt.name))
   
    #Read Data: from dataset and create data loader
    train_dataset = CPDataset(opt)
    train_loader = CPDataLoader(opt, train_dataset)

    #Visualization
    if not os.path.exists(opt.tensorboard_dir):
        os.makedirs(opt.tensorboard_dir)
    writer = SummaryWriter(os.path.join(opt.tensorboard_dir, opt.name))

    #Create model, train & save the final checkpoint
    if opt.stage == 'GMM':
        #GMM model
        model = GMM(opt)
        if not opt.checkpoint =='' and os.path.exists(opt.checkpoint):
            load_checkpoint(model, opt.checkpoint)
        train_gmm(opt, train_loader, model, writer, device_id)
        save_checkpoint(model, os.path.join(opt.checkpoint_dir, opt.name, 'gmm_final.pth'))
    elif opt.stage == 'TOM':
        #TOM model
        model = UnetGenerator(25, 4, 6, ngf=64, norm_layer=nn.InstanceNorm2d)
        if not opt.checkpoint =='' and os.path.exists(opt.checkpoint):
            load_checkpoint(model, opt.checkpoint)
        train_tom(opt, train_loader, model, writer, device_id)
        save_checkpoint(model, os.path.join(opt.checkpoint_dir, opt.name, 'tom_final.pth'))
    else:
        raise NotImplementedError('Model [%s] is not implemented' % opt.stage)
        
    #Finish
    print('Finished training %s, nameed: %s!' % (opt.stage, opt.name))

if __name__ == "__main__":
    main()


Overwriting train.py


In [5]:
%%writefile /kaggle/working/hcmus_cvi_cp-vton/run_train.sh
torchrun \
  --standalone \
  --nnodes=1 \
  --nproc_per_node=2 \
  --rdzv_id=100 \
  --rdzv_backend=c10d \
  --rdzv_endpoint=localhost:29400 \
  train.py \
    --dataroot='/kaggle/input/vton-cp-resized/viton_resize' --name='GMM' --stage='GMM' --workers=2 --checkpoint_dir='/kaggle/working/' --save_count=10000 --keep_step=5000 --decay_step=5000

Writing /kaggle/working/hcmus_cvi_cp-vton/run_train.sh


In [6]:
import torch
!bash run_train.sh

[2024-05-14 09:39:26,781] torch.distributed.run: [WARNING] master_addr is only used for static rdzv_backend and when rdzv_endpoint is not specified.
[2024-05-14 09:39:26,781] torch.distributed.run: [WARNING] 
[2024-05-14 09:39:26,781] torch.distributed.run: [WARNING] *****************************************
[2024-05-14 09:39:26,781] torch.distributed.run: [WARNING] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
[2024-05-14 09:39:26,781] torch.distributed.run: [WARNING] *****************************************
2024-05-14 09:39:34.727196: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-05-14 09:39:34.727206: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to registe

In [7]:
%%writefile /kaggle/working/hcmus_cvi_cp-vton/run_testgmm.sh
torchrun \
  --standalone \
  --nnodes=1 \
  --nproc_per_node=1 \
  --rdzv_id=100 \
  --rdzv_backend=c10d \
  --rdzv_endpoint=localhost:29400 \
  test.py \
    --dataroot='/kaggle/input/vton-cp-resized/viton_resize' --name='GMM' --stage='GMM' --workers=1 --checkpoint='checkpoints/GMM/gmm_final.pth' --data_list='/kaggle/input/vton-cp-resized/viton_resize/test_pairs.txt' --datamode='train'

Writing /kaggle/working/hcmus_cvi_cp-vton/run_testgmm.sh


In [ ]:
!bash run_testgmm.sh

[2024-05-14 10:06:01,829] torch.distributed.run: [WARNING] master_addr is only used for static rdzv_backend and when rdzv_endpoint is not specified.
2024-05-14 10:06:06.123460: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-05-14 10:06:06.123525: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-05-14 10:06:06.125150: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Start to test stage: GMM, named: GMM!
2997it [31:56,  1.53it/s]

In [ ]:
import shutil
shutil.make_archive("output", 'zip', "/kaggle/working/hcmus_cvi_cp-vton/data/test")